# Try fan-out/fan-in on real orders

**Fan-out/fan-in** means splitting one job into tasks that run at the same time, then combining the results into one answer.

In this notebook, a customer asks "Where's my order?" The answer is spread across three systems, so we ask all three at once and combine what they say. We use about 99,000 real orders from the public Olist e-commerce dataset.

### Do I need an API key?

No. Most of this notebook is plain Python, so it runs for free.

| Part | Without a key | With a key |
|---|---|---|
| **Fan-out** | Works | Works the same way |
| **Fan-in** | Fixed sentence rules write the answer | An AI model writes the answer |
| **Voting** | Skipped | Three AI reviewers read a real complaint |
| **Dynamic fan-out** | Works | Works the same way |
| **Real agent** | Skipped | The model chooses the steps |

In Steps 2 to 9, the AI only changes the last step. Fan-out/fan-in is an architecture pattern, and it works with or without a model. Step 10 goes further and lets the model decide the steps.

Run the cells from top to bottom. You need Python 3.10 or later if you want to use an AI model.

---

## Step 1: Install the packages

This cell installs the libraries the notebook needs.

**To use an AI model (optional):** uncomment the lines below and add your key. You can also change the provider, for example `MODEL = "anthropic:claude-opus-5-5"` with `ANTHROPIC_API_KEY` and `langchain-anthropic`.

**Don't save this notebook with a real key in it,** or the key could end up on GitHub.

In [9]:
%pip install -q langchain langgraph pandas kagglehub

# Optional: uncomment these lines to use an AI model.
# Your key goes in the box that appears, so it's never saved in this notebook.
# %pip install -q langchain-openai
import os, getpass
os.environ["MODEL"] = "gpt-4o-mini"
os.environ["OPENAI_API_KEY"] = ""

Note: you may need to restart the kernel to use updated packages.


## Step 2: Import the tools and pick a model

This cell loads the libraries we need. The important part is the last line:

- If `MODEL` is set, `model` becomes a real AI model.
- If `MODEL` isn't set, `model` is `None`, and the notebook uses free rules instead.

That one line is why the notebook runs without a key.

In [10]:
import argparse
import asyncio
import operator
import os
import time
from collections import Counter
from typing import Annotated, Literal, Optional, TypedDict

import kagglehub
import pandas as pd
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from pydantic import BaseModel

# Any provider works, for example MODEL=anthropic:claude-opus-5-5 or MODEL=openai:gpt-5.
model = init_chat_model(os.environ["MODEL"]) if os.environ.get("MODEL") else None

## Step 3: Load the real data

This cell downloads the Olist dataset, about 43 MB, the first time you run it. You don't need a Kaggle account.

It then loads five tables:

| Table | What's in it |
|---|---|
| `orders` | Order status and delivery dates |
| `payments` | How much was paid |
| `items` | Which products and sellers are in each order |
| `sellers` | Where each seller is based |
| `reviews` | What customers wrote |

The small `day()` helper turns a date into plain text, or `None` if the date is missing.

You may see a warning about the `kagglehub` version. You can ignore it.

In [11]:
# Load the dataset. No Kaggle account is needed, and the download is about 43 MB.
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")


def read(table: str, **kwargs) -> pd.DataFrame:
    return pd.read_csv(f"{path}/olist_{table}_dataset.csv", **kwargs)


orders = read("orders", parse_dates=[
    "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]).set_index("order_id")
payments = read("order_payments")
items = read("order_items", parse_dates=["shipping_limit_date"])
sellers = read("sellers").set_index("seller_id")
reviews = read("order_reviews")


def day(value) -> Optional[str]:
    return None if pd.isna(value) else str(value.date())

## Step 4: Create the three systems

In a real company, these would be three separate systems owned by different teams. Here, each function reads only the data that one system would know:

| Function | Pretends to be | What it knows |
|---|---|---|
| `store_system` | The store | Status, approval date, amount paid |
| `warehouse_system` | The warehouse | Items, sellers, ship-by date |
| `courier_system` | The courier | Collected, delivered and estimated dates |

Looking up a table is instant, so `LATENCY` adds a small delay to act like a real API.

`source()` wraps each system with the most important protection in this notebook. It adds a timeout and catches errors. If a system fails, the task returns `"unavailable"`, so one failure can't crash the whole request.

`DOWN` lets us switch a system off on purpose. We'll use it in Step 7.

In [12]:
# Each function reads only what one real system would know.
def store_system(order_id: str) -> dict:
    order = orders.loc[order_id]
    paid = payments[payments.order_id == order_id]
    return {
        "status": order.order_status,
        "approved_on": day(order.order_approved_at),
        "paid": round(float(paid.payment_value.sum()), 2),
    }


def warehouse_system(order_id: str) -> dict:
    lines = items[items.order_id == order_id]
    return {
        "items": len(lines),
        "sellers": lines.seller_id.nunique(),
        "ship_by": day(lines.shipping_limit_date.max()) if len(lines) else None,
    }


def courier_system(order_id: str) -> dict:
    order = orders.loc[order_id]
    return {
        "collected_on": day(order.order_delivered_carrier_date),
        "delivered_on": day(order.order_delivered_customer_date),
        "estimated_on": day(order.order_estimated_delivery_date),
    }


# Local lookups are instant, so this adds the delay a real API would have.
LATENCY = {"store": 0.2, "warehouse": 0.3, "courier": 0.5}
DOWN: set[str] = set()  # Add a source name here to simulate an outage.


def source(name: str, system, timeout: float = 2.0) -> RunnableLambda:
    """Wrap a system so one failure cannot sink the whole request."""

    async def call(inputs: dict) -> dict:
        try:
            if name in DOWN:
                raise ConnectionError(f"{name} is down")
            await asyncio.wait_for(asyncio.sleep(LATENCY[name]), timeout)
            return {"status": "ok", "data": system(inputs["order_id"])}
        except Exception as exc:
            return {"status": "unavailable", "reason": type(exc).__name__}

    return RunnableLambda(call)

## Step 5: Build the fan-out and fan-in

This is the core of the pattern.

- **`fan_out`** is the fan-out. `RunnableParallel` sends the order to all three systems at the same time.
- **`fan_in_prompt`** is the instruction for an AI model. It says to use only these results and never to guess.
- **`rule_based_answer()`** is the free fan-in. It uses fixed rules to turn the results into a sentence.
- **`where_is_my_order()`** runs everything: it fans out, prints the results and timing, then fans in.

This line chooses how the fan-in works:

```python
if model:
    # An AI model writes the reply.
else:
    # The free rules write the reply.
```

Nothing visible happens when you run this cell. It only defines the code.

In [13]:
# Sectioning: different questions, different systems.
fan_out = RunnableParallel(
    store=source("store", store_system),
    warehouse=source("warehouse", warehouse_system),
    courier=source("courier", courier_system),
)

fan_in_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an order support assistant. Answer the customer's question "
     "using only the results below. If a source is unavailable, say what "
     "you could not confirm. Never guess.\n\nResults:\n{results}"),
    ("human", "{question}"),
])


def rule_based_answer(results: dict) -> str:
    """A free fan-in with no model, so anyone can run this."""
    data = {name: r["data"] for name, r in results.items() if r["status"] == "ok"}
    store, warehouse, courier = (data.get(n) for n in ("store", "warehouse", "courier"))
    missing = [n for n in ("store", "warehouse", "courier") if n not in data]
    parts = []

    if store and store["status"] in ("canceled", "unavailable"):
        label = "was canceled" if store["status"] == "canceled" else "could not be fulfilled"
        parts.append(f"This order {label}.")
    elif courier and courier["delivered_on"]:
        days = (pd.Timestamp(courier["delivered_on"]) - pd.Timestamp(courier["estimated_on"])).days
        timing = f"{days} days late" if days > 0 else "on time"
        parts.append(f"Your order was delivered on {courier['delivered_on']}, {timing}.")
    elif courier and courier["collected_on"]:
        parts.append(f"The courier collected your order on {courier['collected_on']} "
                     f"and estimates delivery by {courier['estimated_on']}.")
    elif courier:
        parts.append("The courier has not collected your order yet.")
    elif store:
        parts.append(f"The store shows your order as {store['status']}.")

    if warehouse and warehouse["items"]:
        parts.append(f"It has {warehouse['items']} item(s) from "
                     f"{warehouse['sellers']} seller(s).")
    if missing:
        parts.append(f"I could not confirm details from the {' or '.join(missing)} system.")
    return " ".join(parts) or "I could not confirm anything about this order right now."


async def where_is_my_order(order_id: str) -> None:
    question = "I paid for my order. Where is it?"
    started = time.perf_counter()
    results = await fan_out.ainvoke({"order_id": order_id})
    elapsed = time.perf_counter() - started

    for name, result in results.items():
        print(f"{name:>10}: {result.get('data', result)}")
    print(f"Fan-out took {elapsed:.2f}s. One after another: {sum(LATENCY.values()):.2f}s.")

    if model:
        chain = fan_in_prompt | model | StrOutputParser()
        reply = await chain.ainvoke({"results": results, "question": question})
    else:
        reply = rule_based_answer(results)
    print(f"\nFan-in answer: {reply}")

## Step 6: Pick some real orders to try

`sample_orders()` finds one real order for each situation, so you have interesting cases to test.

In [14]:
def sample_orders() -> dict[str, str]:
    """One real order for each situation, so you have something to try."""
    multi = items.groupby("order_id").seller_id.nunique()
    delivered = orders[orders.order_status == "delivered"].dropna(
        subset=["order_delivered_customer_date", "order_estimated_delivery_date"])
    late = (delivered.order_delivered_customer_date.dt.normalize()
            > delivered.order_estimated_delivery_date)
    picks = {
        "Delivered on time": delivered[~late].index,
        "Delivered late": delivered[late].index,
        "With the courier": orders[orders.order_status == "shipped"].index,
        "Not collected yet": orders[orders.order_status == "processing"].index,
        "Canceled": orders[orders.order_status == "canceled"].index,
        "Several sellers": multi[multi >= 3].index,
    }
    return {label: sorted(ids)[0] for label, ids in picks.items()}

In [15]:
samples = sample_orders()
samples

{'Delivered on time': '00010242fe8c5a6d1ba2dd792cb16214',
 'Delivered late': '000e906b789b55f64edcb1f84030f90d',
 'With the courier': '002f19a65a2ddd70a090297872e6d64e',
 'Not collected yet': '042ce41927758767ec2aa0de06486b4c',
 'Canceled': '00310b0c75bb13015ec4d82d341865a4',
 'Several sellers': '06d5cbe68295558fd6c48998db3ea397'}

The output shows each situation with its real order ID.

## Step 7: Run the fan-out

This runs the whole pattern on a real order. Look for three things in the output:

1. The result from each system.
2. The timing. The tasks run at the same time, so the total is about **0.5 seconds** rather than 1.0 second.
3. The fan-in answer that combines everything.

**Try it:** change `"Delivered late"` to another label, such as `"Canceled"` or `"With the courier"`.

In [16]:
await where_is_my_order(samples["Delivered late"])

     store: {'status': 'delivered', 'approved_on': '2017-11-21', 'paid': 33.84}
 warehouse: {'items': 1, 'sellers': 1, 'ship_by': '2017-11-27'}
   courier: {'collected_on': '2017-11-22', 'delivered_on': '2017-12-09', 'estimated_on': '2017-12-07'}
Fan-out took 0.50s. One after another: 1.00s.

Fan-in answer: Your order was approved on November 21, 2017, and marked as delivered. However, if you are asking for the current status or location of the order in 2023, I could not confirm that information.


### What if a system fails?

This cell switches off the courier system, runs the same request, then switches it back on.

Look at the answer. The request still finishes, and the assistant says what it couldn't confirm. This is called a **partial answer**, and it's what `source()` from Step 4 makes possible.

**Try it:** change `"courier"` to `"store"` or `"warehouse"`.

In [17]:
DOWN.add("courier")
await where_is_my_order(samples["With the courier"])
DOWN.clear()

     store: {'status': 'shipped', 'approved_on': '2018-03-21', 'paid': 77.29}
 warehouse: {'items': 1, 'sellers': 1, 'ship_by': '2018-03-27'}
   courier: {'status': 'unavailable', 'reason': 'ConnectionError'}
Fan-out took 0.31s. One after another: 1.00s.

Fan-in answer: Your order has been shipped. It was approved on March 21, 2018, and the payment was $77.29. It is scheduled to be shipped by March 27, 2018.


## Step 8: Set up voting

So far, each task has asked a **different** question. That's called **sectioning**. It works when the facts are in different places.

**Voting** is the other way to use the pattern. Every task answers the **same** question, and we compare the answers. It works when one fact is hard to interpret.

Here, we use real orders marked as delivered where the customer wrote "não recebi" ("I didn't receive it").

- **`Vote`** forces each reviewer to answer `yes`, `no` or `unclear`, so the answers can be counted.
- **`LENSES`** gives each reviewer a different angle, because identical reviewers tend to make the same mistake.
- **`disputed_reviews()`** finds these conflicting orders.
- **`resolve_by_vote()`** runs the vote and decides what to do:

| Result | Decision |
|---|---|
| Reviewers disagree | Trust the delivery record |
| All say "not received", but the record says delivered | Send it to a person |
| Otherwise | Accept the unanimous answer |

In [18]:
# Voting: same question, three reviewers, on real customer reviews in Portuguese.
class Vote(BaseModel):
    received: Literal["yes", "no", "unclear"]
    reason: str


review_prompt = ChatPromptTemplate.from_messages([
    ("system", "You review customer comments, which are in Portuguese. {lens}"),
    ("human", "Review: {review}\n\nDid the customer receive their order?"),
])

LENSES = [
    "Read the comment literally. Do not infer anything it does not state.",
    "Focus on whether any part of the order is said to be missing.",
    "Look for wording that could mean a late, partial or wrong delivery.",
]


def disputed_reviews(limit: int = 5) -> pd.DataFrame:
    """Orders marked delivered where the customer says they did not receive them."""
    said_no = reviews.review_comment_message.str.contains(
        "não recebi|nao recebi", case=False, na=False)
    marked = orders.index[orders.order_status == "delivered"]
    return reviews[said_no & reviews.order_id.isin(marked)].sort_values("review_id").head(limit)


async def resolve_by_vote(order_id: str, review: str) -> dict:
    if model is None:
        return {"skipped": "Voting needs an AI model. Set MODEL to turn it on."}
    reviewer = review_prompt | model.with_structured_output(Vote, method="json_schema")

    # Fan-out: same question, three reviewers.
    votes = await reviewer.abatch([{"lens": lens, "review": review} for lens in LENSES])

    # Fan-in: compare the votes, then compare them with the delivery record.
    tally = Counter(vote.received for vote in votes)
    answer, count = tally.most_common(1)[0]
    delivered = courier_system(order_id)["delivered_on"] is not None
    if count < len(votes):
        return {"votes": dict(tally), "received": delivered, "decided_by": "delivery record"}
    if answer == "no" and delivered:
        return {"votes": dict(tally), "received": "disputed", "decided_by": "send to a person"}
    return {"votes": dict(tally), "received": answer, "decided_by": "unanimous vote"}

### Run the vote

This prints a real customer review, then runs the vote.

**Without a key,** you'll see a `skipped` message. That's expected, because voting needs a real model. With a key, each reviewer makes one paid model call.

In [19]:
case = disputed_reviews().iloc[0]
print(case.review_comment_message)
await resolve_by_vote(case.order_id, case.review_comment_message)

não recebi o produto 


/Users/rishan/Library/Python/3.9/lib/python/site-packages/pydantic/main.py:364: UserWarning: Pydantic serializer warnings:
  Expected `none` but got `Vote` - serialized value may not be as expected
  return self.__pydantic_serializer__.to_python(


{'votes': {'no': 3}, 'received': 'disputed', 'decided_by': 'send to a person'}

## Step 9: Set up dynamic fan-out

So far, we've known the tasks in advance. Sometimes the data decides. One order might have one seller, and the next might have three.

**Dynamic fan-out** creates one task per seller at runtime, using LangGraph:

| Piece | What it does |
|---|---|
| `load_sellers` | Finds the sellers in this order |
| `fan_out_sellers` | Creates one `Send` task per seller |
| `check_seller` | Does the work for one seller |
| `combine_sellers` | The fan-in: combines every seller's report |

Parallel tasks all write to `seller_reports`. `operator.add` tells LangGraph to join their lists together.

The last few lines connect the pieces into a graph, then `compile()` makes it ready to run.

In [20]:
# Dynamic fan-out: one task per seller, and only the data decides how many.
class OrderState(TypedDict):
    order_id: str
    seller_ids: list[str]
    seller_reports: Annotated[list[dict], operator.add]
    summary: str


class SellerState(TypedDict):
    order_id: str
    seller_id: str


def load_sellers(state: OrderState) -> dict:
    lines = items[items.order_id == state["order_id"]]
    return {"seller_ids": sorted(lines.seller_id.unique())}


def fan_out_sellers(state: OrderState) -> list[Send]:
    return [Send("check_seller", {"order_id": state["order_id"], "seller_id": s})
            for s in state["seller_ids"]]


async def check_seller(state: SellerState) -> dict:
    await asyncio.sleep(0.3)
    lines = items[(items.order_id == state["order_id"]) & (items.seller_id == state["seller_id"])]
    seller = sellers.loc[state["seller_id"]]
    return {"seller_reports": [{
        "city": seller.seller_city,
        "items": len(lines),
        "ship_by": day(lines.shipping_limit_date.max()),
    }]}


def combine_sellers(state: OrderState) -> dict:
    reports = sorted(state["seller_reports"], key=lambda r: r["ship_by"])
    lines = [f"{r['items']} item(s) from {r['city']}, ship by {r['ship_by']}" for r in reports]
    return {"summary": f"{len(reports)} sellers: " + "; ".join(lines)}


builder = StateGraph(OrderState)
builder.add_node("load_sellers", load_sellers)
builder.add_node("check_seller", check_seller)
builder.add_node("combine_sellers", combine_sellers)
builder.add_edge(START, "load_sellers")
builder.add_conditional_edges("load_sellers", fan_out_sellers, ["check_seller"])
builder.add_edge("check_seller", "combine_sellers")
builder.add_edge("combine_sellers", END)
graph = builder.compile()

### Run the dynamic fan-out

This runs the graph on a real order from several sellers. It creates one task per seller and combines the results. It needs no model.

In [21]:
state = await graph.ainvoke({"order_id": samples["Several sellers"]})
print(state["summary"])

3 sellers: 1 item(s) from sao paulo, ship by 2018-04-02; 1 item(s) from atibaia, ship by 2018-04-02; 1 item(s) from sao paulo, ship by 2018-04-03


## Step 10: Let the model decide (a real agent)

Everything so far has been a **workflow**. Our code decided every step, and it always called the same systems. The model only wrote the reply or cast a vote.

In an **agent**, the model decides the steps. It chooses which tools to call, reads the results, and decides what to do next.

| | Workflow (Steps 5 to 9) | Agent (this step) |
|---|---|---|
| **Who decides the steps?** | Our code | The model |
| **Which systems are called?** | Always all three | Only the ones the model needs |
| **Can it adapt to results?** | No | Yes |
| **Needs an API key?** | No | Yes |

This cell has three parts:

- **The tools:** each `@tool` wraps a system from Step 4. The model reads each description to decide what to call.
- **`run_tool()`:** runs one tool. A failure becomes a message the model can read, so it can't crash the agent.
- **`run_agent()`:** the agent itself, which is one loop:

1. The model decides: call some tools, or answer.
2. **Fan-out:** `asyncio.gather` runs the chosen tools at the same time.
3. **Fan-in:** every result goes back to the model.
4. The loop repeats until the model answers, or until `MAX_TURNS` stops it.

Nothing visible happens when you run this cell. It only defines the agent.

In [22]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool

MAX_TURNS = 5  # Stops the agent if it never finishes.
TOOL_TIMEOUT = 2  # Seconds to wait for one tool.


@tool
async def check_store(order_id: str) -> str:
    """Check an order's status, approval date and amount paid."""
    await asyncio.sleep(LATENCY["store"])
    return str(store_system(order_id))


@tool
async def check_warehouse(order_id: str) -> str:
    """Check an order's items, sellers and ship-by date."""
    await asyncio.sleep(LATENCY["warehouse"])
    return str(warehouse_system(order_id))


@tool
async def check_courier(order_id: str) -> str:
    """Check when an order was collected, and its delivery dates."""
    await asyncio.sleep(LATENCY["courier"])
    return str(courier_system(order_id))


@tool
async def check_cancellation_policy() -> str:
    """Look up the rules for cancelling an order."""
    await asyncio.sleep(0.2)
    return "An order can be cancelled for free until the courier collects it."


TOOLS = {t.name: t for t in [check_store, check_warehouse, check_courier, check_cancellation_policy]}

INSTRUCTIONS = (
    "You are an order support assistant. Use the tools to answer the customer. "
    "Call only the tools you need. When tools don't depend on each other, call "
    "them together in one step. Use only what the tools tell you, and say what "
    "you could not confirm."
)


async def run_tool(call: dict) -> ToolMessage:
    """Run one tool. A failure becomes a message the model can read."""
    try:
        chosen = TOOLS[call["name"]]
        result = await asyncio.wait_for(chosen.ainvoke(call["args"]), TOOL_TIMEOUT)
    except Exception as exc:
        result = f"This tool failed: {type(exc).__name__}"
    return ToolMessage(content=result, tool_call_id=call["id"])


async def run_agent(question: str) -> str:
    if model is None:
        return "Skipped: the agent needs an AI model. Set MODEL in Step 1 to turn it on."

    agent_model = model.bind_tools(list(TOOLS.values()))
    messages = [SystemMessage(INSTRUCTIONS), HumanMessage(question)]

    for turn in range(1, MAX_TURNS + 1):
        # The model decides: call some tools, or answer.
        response = await agent_model.ainvoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return StrOutputParser().invoke(response)

        # Fan-out: run every tool the model chose at the same time.
        names = [call["name"] for call in response.tool_calls]
        print(f"Turn {turn}: the model chose {', '.join(names)}")
        started = time.perf_counter()
        results = await asyncio.gather(*(run_tool(call) for call in response.tool_calls))
        print(f"        {len(results)} tool(s) finished in {time.perf_counter() - started:.1f}s")

        # Fan-in: give every result back to the model.
        messages.extend(results)

    return "I could not finish this request. Please ask a person to help."

### Ask a question that needs several steps

The model has to work out where the order is, then decide whether the cancellation rules matter. Your code doesn't tell it which tools to use.

Look at the `Turn` lines to see what it chose. Different models may choose different steps.

**Without a key,** you'll see a `Skipped` message. With a key, each turn makes one paid model call.

In [23]:
order_id = samples["Not collected yet"]
print(await run_agent(f"Where's my order {order_id}, and can I still cancel it?"))

Turn 1: the model chose check_store, check_cancellation_policy
        2 tool(s) finished in 0.2s
Turn 2: the model chose check_courier
        1 tool(s) finished in 0.5s
Your order with ID **042ce41927758767ec2aa0de06486b4c** is currently in **processing** status and was approved on **December 3, 2017**, with a payment amount of **$592.77**. 

Since the order has not yet been collected by the courier, you can still cancel it for free.


### Ask a smaller question

This only asks about payment, so the model should need just `check_store`. A workflow would still call all three systems.

**Try it:** write your own question, or use an order ID from Step 6.

In [24]:
print(await run_agent(f"How much did I pay for order {order_id}?"))

Turn 1: the model chose check_store
        1 tool(s) finished in 0.2s
You paid $592.77 for order 042ce41927758767ec2aa0de06486b4c.


---

## What you've learned

- **Fan-out** runs independent tasks at the same time.
- **Fan-in** combines their results into one answer.
- **Sectioning** asks different questions. **Voting** asks the same question.
- **Dynamic fan-out** lets the data decide how many tasks to run.
- **Failures** should produce a partial answer, not a crash.
- A **workflow** lets your code decide the steps. An **agent** lets the model decide.

## Try next

1. Change the order labels in Step 7.
2. Take a different system offline.
3. Add an API key in Step 1, then compare the AI answer with the free one.
4. Ask the agent in Step 10 your own question.

*Data: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce), CC BY-NC-SA 4.0.*